# Smith-Waterman: C vs CUDA - Week 7 setup
CEPARCO Integrating Project, Group 3

Checks that the Colab environment works for the project: GPU, `nvcc`, a tiny CUDA
compile-and-run test, the repository, and the test/benchmark input files.

**Before running:** Runtime -> Change runtime type -> choose a **GPU** (usually a T4).

## 1. GPU and compiler

In [ ]:
import os
os.environ["PATH"] += ":/usr/local/cuda/bin"      # in case nvcc is not already on PATH
!nvidia-smi
!nvcc --version
!gcc --version | head -1

## 2. CUDA smoke test
Compiles and runs a 1,000-thread kernel with the same flags the project will use
(`-arch=native` lets `nvcc` detect the GPU; needs CUDA 11.5 or newer - otherwise use e.g. `-arch=sm_75`).
Expected output: `sum=499500 (expected 499500) -> OK`

In [ ]:
%%writefile hello.cu
#include <cstdio>
__global__ void fill(int *x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) x[i] = i;
}
int main() {
    const int n = 1000;
    int *d, h[1000];
    cudaMalloc(&d, n * sizeof(int));
    fill<<<(n + 255) / 256, 256>>>(d, n);
    cudaMemcpy(h, d, sizeof h, cudaMemcpyDeviceToHost);
    long s = 0;
    for (int i = 0; i < n; i++) s += h[i];
    printf("sum=%ld (expected 499500) -> %s\n", s, s == 499500 ? "OK" : "WRONG");
    cudaFree(d);
    return s == 499500 ? 0 : 1;
}

In [ ]:
!nvcc -O2 -arch=native hello.cu -o hello && ./hello

## 3. Get the repository
Option A (recommended): clone the team repository - **edit `REPO_URL` first**.
Option B: upload `smith-waterman-week7.zip` (uncomment the lines).

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # <-- edit this

!git clone $REPO_URL smith-waterman
%cd smith-waterman

# --- Option B: upload the zip instead ---
# from google.colab import files
# files.upload()                       # choose smith-waterman-week7.zip
# !unzip -q -o smith-waterman-week7.zip
# %cd smith-waterman

## 4. Validate the test sequences
Runs an independent pure-Python Smith-Waterman on the four small test inputs and compares
with the expected scores, coordinates, alignments and (for the worked example) the full
score matrix; also checks the format of the benchmark inputs.

In [ ]:
!python3 scripts/check_expected.py

## 5. Benchmark inputs are reproducible
Regenerates each benchmark file from its seed and compares it with the committed copy.
All three lines should say `identical`.

In [ ]:
for n in (1000, 5000, 10000):
    !python3 scripts/gen_sequences.py --length $n --seed 1 --out /tmp/regen_$n.fasta > /dev/null
    !cmp -s /tmp/regen_$n.fasta data/bench_$n.fasta && echo "bench_$n: identical" || echo "bench_$n: DIFFERENT (check the Python version)"

## 6. Record the environment
Colab's GPU depends on availability. Paste this line into the README / report so every
timing can be traced to the machine that produced it.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | tail -1
!python3 --version